# Copy the NWM 3.0 retrospective (hourly streamflow) from legacy

| v1 dataset (`public`) | what |
| --- | --- |
| `nwm30_retrospective` | simulated hourly streamflow at the gages' `nwm30` reaches, 1981–2023 (legacy `secondary_timeseries`, `nwm30_retrospective`, variable `streamflow_hourly_inst`: 3.06 B rows, 8,299 reaches) |

It shares `nwm30_medium_range_series`, so the `usgs_nwm30` crosswalk pairs it with
`usgs_inst_streamflow` (`pair: {relation: usgs_nwm30}`). Legacy's daily means of it (543 reaches)
and its basin forcing (the `primary` rows, `usgsbasin-*`) are not copied.

**Run it as a platform admin, on use1.** Spark writes the rows straight into the `public`
Iceberg table, a month at a time (the API takes at most 64 MB per write): a one-time admin
migration, as notebooks 05 and 06.

**Re-runnable:** registering skips a dataset that exists; series are added, never removed; each
month copied replaces that month (`START` resumes from a month). Start with **2. Count**: it
writes nothing.

In [ ]:
import os

import pandas as pd
import requests
import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

teehr.__version__

## 1. Settings

In [ ]:
DATASET = "nwm30_retrospective"
LEGACY_CONFIG, VARIABLE = "nwm30_retrospective", "streamflow_hourly_inst"
SERIES = "nwm30_medium_range_series"  # shared: the usgs_nwm30 crosswalk's right side
SOURCE = ("NOAA National Water Model 3.0 retrospective (CHRTOUT); copied from the legacy TEEHR "
          "warehouse")
START = None  # e.g. "2001-06" to resume the copy from a month

CONTROL_PLANE = os.environ.get(
    "CONTROL_PLANE_URL", "http://teehr-control-plane.teehr-hub.svc.cluster.local:8000"
)
VERIFY_TLS = "local.app.garden" not in CONTROL_PLANE  # KinD's certificate is self-signed
LEGACY, PUBLIC = "iceberg.teehr", "iceberg.public"

In [ ]:
# A cluster for the copy (~500 months, ~6 M rows each).
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=16,
    executor_memory="16g",
    executor_cores=4,
)
spark.conf.set("spark.sql.iceberg.check-nullability", "false")
spark.conf.set("spark.sql.session.timeZone", "UTC")  # months are UTC, as Iceberg's partitions

def access_token():
    """Your access token, refreshed when it's about to expire (JupyterHub sets it)."""
    try:
        from teehr.utils.keycloak_auth import refresh_user_token_from_env
    except ImportError:  # a kernel without the helper: the token as JupyterHub set it
        return os.environ["POLARIS_USER_TOKEN"]
    access, _refresh = refresh_user_token_from_env()
    return access


api = requests.Session()
api.verify = VERIFY_TLS


def call(method, path, **kwargs):
    resp = api.request(method, f"{CONTROL_PLANE}/v1{path}",
                       headers={"Authorization": f"Bearer {access_token()}"}, **kwargs)
    if resp.status_code >= 400 and resp.status_code != 404:
        raise RuntimeError(f"{method} {path}: HTTP {resp.status_code}: {resp.text[:1000]}")
    return resp


def exists(name):
    return call("GET", f"/projects/public/datasets/{name}").status_code == 200


def register(body):
    if exists(body["name"]):
        print(f"public.{body['name']}: exists")
        return
    call("POST", "/projects/public/datasets", json=body)
    print(f"public.{body['name']}: registered")


def month_range(column, month):
    """A time range filter for one UTC month (prunes the scan, unlike date_format)."""
    start = pd.Period(month, freq="M")
    return (f"{column} >= TIMESTAMP '{start.start_time:%Y-%m-%d} 00:00:00' "
            f"AND {column} < TIMESTAMP '{(start + 1).start_time:%Y-%m-%d} 00:00:00'")


def per_month(table, name, where="TRUE"):
    return spark.sql(f"""
        SELECT date_format(value_time, 'yyyy-MM') AS month, count(*) AS {name}
        FROM {table} WHERE {where} AND value IS NOT NULL GROUP BY 1
    """).toPandas()


me = call("GET", "/projects/public").json()
print(f"public: {me['status']}; your role there: {me.get('my_role')}")

## 2. Count (writes nothing)

Rows per month in legacy, units, and the reaches.

In [ ]:
%%time
LEGACY_ROWS = (f"{LEGACY}.secondary_timeseries WHERE configuration_name = '{LEGACY_CONFIG}' "
               f"AND variable_name = '{VARIABLE}'")
legacy = per_month(f"{LEGACY}.secondary_timeseries", "legacy",
                   f"configuration_name = '{LEGACY_CONFIG}' AND variable_name = '{VARIABLE}'")
legacy = legacy.sort_values("month").reset_index(drop=True)
print(f"{len(legacy)} months, {legacy.month.min()} to {legacy.month.max()}, "
      f"{int(legacy.legacy.sum()):,} rows")
spark.sql(f"""
    SELECT unit_name, count(DISTINCT location_id) AS reaches, count(*) AS rows
    FROM {LEGACY_ROWS} AND {month_range('value_time', legacy.month.max())} GROUP BY 1
""").show()

## 3. Register (as you) and add the series

`simulated`, `inst`, period `instant`, no `reference_time`; the shared series table. Its
reaches are added to the series table (missing ones only).

In [ ]:
register({
    "type": "timeseries", "name": DATASET, "role": "simulated", "variable": "streamflow",
    "statistic": "inst", "series_table": SERIES, "source": SOURCE,
    "description": "NWM 3.0 retrospective hourly streamflow at the gages' reaches, 1981-2023",
})
spark.sql(f"""
    MERGE INTO {PUBLIC}.{SERIES} t
    USING (SELECT DISTINCT location_id AS series_key FROM {LEGACY_ROWS}) s
    ON t.series_key = s.series_key
    WHEN NOT MATCHED THEN INSERT (series_key, location_key, location_layer, period, resolution)
        VALUES (s.series_key, NULL, NULL, NULL, NULL)
""")
print(f"{SERIES}: {spark.table(f'{PUBLIC}.{SERIES}').count():,} series")

## 4. Copy, a month at a time

Each month replaces that month of the v1 table. Values are rounded to 0.01 m3/s (the model's
output precision; legacy stored float32).

In [ ]:
%%time
for month in legacy.month:
    if START and month < START:
        continue
    spark.sql(f"""
        SELECT location_id AS series_key, value_time, ROUND(CAST(value AS DOUBLE), 2) AS value
        FROM {LEGACY_ROWS} AND {month_range('value_time', month)} AND value IS NOT NULL
    """).writeTo(f"{PUBLIC}.{DATASET}").overwritePartitions()
    print(f"{month}: copied")

## 5. Check: v1 against legacy, month by month

In [ ]:
%%time
both = legacy.merge(per_month(f"{PUBLIC}.{DATASET}", "v1"), on="month", how="outer").fillna(0)
both["matches"] = both.legacy == both.v1
print(f"{DATASET}: {int(both.v1.sum()):,} rows; every month matches: {both.matches.all()}")
display(both[~both.matches])